# Puma to CZ1990

In [1]:
import os
import pandas as pd
import numpy as np 

In [2]:
os.chdir("/Users/mengsong/Library/CloudStorage/Dropbox/research/wfh/cz/cz90")

## Puma1990 to CZ1990

In [3]:
puma1990 = pd.read_csv('data/geocorr1990_13JAN1656174.csv')
puma1990['county_fips'] = puma1990['county'].astype(str).str.zfill(5)
puma1990.head()

,STATE CODE (FIPS),PUMA code from 1990 5% (A) Sample,county,1990 POP,apuma to county allocation factor,county_fips
0,1,100,1033,51666,0.3934,01033
1,1,100,1077,79661,0.6066,01077
2,1,200,1059,27814,0.2366,01059
3,1,200,1127,67670,0.5757,01127
4,1,200,1133,22053,0.1876,01133


In [4]:
# Verify allocation factors sum to 1 for each PUMA 1990
puma1990.groupby(['STATE CODE (FIPS)', 'PUMA code from 1990 5% (A) Sample'])[
    'apuma to county allocation factor'
].sum().describe()

count    1726.000000
mean        1.000000
std         0.000035
min         0.999800
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000200
Name: apuma to county allocation factor, dtype: float64

In [5]:
county90_cz90 = pd.read_csv('data/counties90-29c6g4u.csv', encoding="latin-1")
county90_cz90['county_fips'] = county90_cz90['FIPS'].astype(str).str.zfill(5)
county90_cz90 = county90_cz90[["county_fips", "ERS90"]]
county90_cz90.head()

,county_fips,ERS90
0,10001,19901
1,10003,19800
2,10005,19901
3,01001,11101
4,01003,11001


In [6]:
print(puma1990['county_fips'].nunique())
print(county90_cz90['county_fips'].nunique())

3141
3141


In [7]:
puma90_cz90 = pd.merge(puma1990, county90_cz90, how="left", on="county_fips")
print(puma90_cz90['ERS90'].isnull().sum())

0


In [8]:
# Construct allocation factor: fraction of PUMA population allocated to each CZ
puma90_cz90_crosswalk = (
    puma90_cz90.groupby(
        ['STATE CODE (FIPS)', 'PUMA code from 1990 5% (A) Sample', 'ERS90']
    )
    .agg({'apuma to county allocation factor': 'sum'})
    .reset_index()
)

# Create combined PUMA 1990 code (state + puma)
puma90_cz90_crosswalk['puma1990'] = (
    puma90_cz90_crosswalk['STATE CODE (FIPS)'].astype(str).str.zfill(2) + 
    puma90_cz90_crosswalk['PUMA code from 1990 5% (A) Sample'].astype(str).str.zfill(4)
)

# Rename and select final columns
puma90_cz90_crosswalk = puma90_cz90_crosswalk.rename(
    columns={
        'ERS90': 'cz1990',
        'apuma to county allocation factor': 'afactor'
    }
)

puma90_cz90_crosswalk = puma90_cz90_crosswalk[['puma1990', 'cz1990', 'afactor']]
puma90_cz90_crosswalk['afactor'] = puma90_cz90_crosswalk['afactor'].round(3)
puma90_cz90_crosswalk

,puma1990,cz1990,afactor
0,010100,6200,1.000
1,010200,6200,0.237
2,010200,10600,0.763
3,010300,10501,0.164
4,010300,10600,0.502
...,...,...,...
2537,560300,36404,0.729
2538,560400,34301,0.286
2539,560400,34302,0.112
2540,560400,34304,0.252


In [9]:
check = puma90_cz90_crosswalk.groupby('puma1990').sum()
print(check.describe())

              cz1990      afactor
count    1726.000000  1726.000000
mean    31796.567787     1.000001
std     37423.838601     0.000206
min       100.000000     0.999000
25%     14928.000000     1.000000
50%     21201.500000     1.000000
75%     37500.000000     1.000000
max    554098.000000     1.002000


In [10]:
# Compare against the Dorn's table
dorn_puma1990_cz1990 = pd.read_stata("data/cw_puma1990_czone.dta")
dorn_puma1990_cz1990['puma1990'] = dorn_puma1990_cz1990['puma1990'].astype(str).str.zfill(6)
dorn_puma1990_cz1990['cz1990'] = dorn_puma1990_cz1990['czone'].astype(int)
dorn_puma1990_cz1990 = dorn_puma1990_cz1990[['puma1990', 'cz1990', 'afactor']]

comparison = pd.merge(
    puma90_cz90_crosswalk,
    dorn_puma1990_cz1990,
    on=['puma1990', 'cz1990'],
    how='outer',
    suffixes=('_mine', '_dorn'),
    indicator=True
)

print(comparison['_merge'].value_counts())
print(f"\nTotal rows: {len(comparison)}")

matched = comparison[comparison['_merge'] == 'both']
print(f"\nMatched pairs: {len(matched)}")
print(f"Mean absolute difference: {(matched['afactor_mine'] - matched['afactor_dorn']).abs().mean():.6f}")
print(f"Max absolute difference: {(matched['afactor_mine'] - matched['afactor_dorn']).abs().max():.6f}")

_merge
both          2542
left_only        0
right_only       0
Name: count, dtype: int64

Total rows: 2542

Matched pairs: 2542
Mean absolute difference: 0.000090
Max absolute difference: 0.002000


In [11]:
puma90_cz90_crosswalk.to_csv("output/puma1990_cz1990_crosswalk.csv", index=False)
puma90_cz90_crosswalk.to_stata("output/puma1990_cz1990_crosswalk.dta", write_index=False)

## Puma2000 to CZ1990

In [13]:
puma2000 = pd.read_csv('data/geocorr2000_14JAN1537489.csv',skiprows=[1],dtype=str)
puma2000.head()

,state,puma5,county,stab,cntyname,pop2k,afact
0,01,00100,01033,AL,Colbert AL,54984,0.3846
1,01,00100,01077,AL,Lauderdale AL,87966,0.6154
2,01,00200,01089,AL,Madison AL,157952,1
3,01,00300,01083,AL,Limestone AL,65676,0.3561
4,01,00300,01089,AL,Madison AL,118748,0.6439


In [14]:
puma2000["pop2k"] = puma2000["pop2k"].astype(float)
puma2000["afact"] = puma2000["afact"].astype(float)

puma2000.groupby(["state", "puma5"])["afact"].sum().describe()

count    2071.000000
mean        1.000000
std         0.000031
min         0.999800
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000200
Name: afact, dtype: float64

In [15]:
print(puma2000['county'].str.len().unique())
print(county90_cz90["county_fips"].str.len().unique())

print(puma2000["county"].nunique())
print(county90_cz90["county_fips"].nunique())

[5]
[5]
3141
3141


In [16]:
# Identify county code mismatches/changes between PUMA 2000 and County-CZ 1990 mapping

puma2000_counties = set(puma2000["county"].unique())
county90_counties = set(county90_cz90["county_fips"].unique())

only_in_puma2000 = puma2000_counties - county90_counties
only_in_county90 = county90_counties - puma2000_counties

print(f"Counties only in PUMA 2000: {len(only_in_puma2000)}")
print(f"Counties only in County-CZ 1990: {len(only_in_county90)}")

if len(only_in_puma2000) > 0:
    print(f"\nCounties in PUMA 2000 but not in CZ mapping:")
    print(sorted(only_in_puma2000))

if len(only_in_county90) > 0:
    print(f"\nCounties in CZ mapping but not in PUMA 2000:")
    print(sorted(only_in_county90))

Counties only in PUMA 2000: 4
Counties only in County-CZ 1990: 4

Counties in PUMA 2000 but not in CZ mapping:
['02068', '02232', '02282', '12086']

Counties in CZ mapping but not in PUMA 2000:
['02231', '12025', '30113', '51780']


In [17]:
# Map PUMA 2000 county codes to their 1990 equivalents
# Source: https://www.census.gov/programs-surveys/geography/technical-documentation/county-changes.January_2020.html

county_changes_2000_to_1990 = {
    '02068': '02290',  # Yukon-Koyukuk (02-290) -> Denali Borough (02-068) (1990)
    '02232': '02231',  # Skagway-Yakutat-Angoon (02-231) -> Skagway-Hoonah-Angoon (02-232) (1992)
    '02282': '02231',  # Skagway-Yakutat-Angoon (02-231) -> Yakutat City and Borough (02-282) (1992)
    '12086': '12025',  # Dade County (12-025) -> Miami-Dade County (12-086) (1997)
}

puma2000['county_1990'] = puma2000['county'].replace(county_changes_2000_to_1990)

In [18]:
puma2000_cz90 = pd.merge(
    puma2000,
    county90_cz90,
    how="left",
    left_on="county_1990",
    right_on="county_fips",
)

print(f"Null CZ codes: {puma2000_cz90['ERS90'].isnull().sum()}")

Null CZ codes: 0


In [19]:
# Construct allocation factor: fraction of PUMA population allocated to each CZ
puma2000_cz90_crosswalk = (
    puma2000_cz90.groupby(['state', 'puma5', 'ERS90'])
    .agg({'afact': 'sum'})
    .reset_index()
)

# Create combined PUMA 2000 code (state + puma, 6 digits total)
puma2000_cz90_crosswalk['puma2000'] = (
    puma2000_cz90_crosswalk['state'].astype(int).astype(str).str.zfill(2) + 
    puma2000_cz90_crosswalk['puma5'].astype(int).astype(str).str.zfill(4)
)

# Rename and select final columns
puma2000_cz90_crosswalk = puma2000_cz90_crosswalk.rename(columns={'ERS90': 'cz1990', 'afact': 'afactor'})
puma2000_cz90_crosswalk = puma2000_cz90_crosswalk[['puma2000', 'cz1990', 'afactor']]
puma2000_cz90_crosswalk['afactor'] = puma2000_cz90_crosswalk['afactor'].round(3)

puma2000_cz90_crosswalk.head()

,puma2000,cz1990,afactor
0,010100,6200,1.000
1,010200,6000,1.000
2,010300,6000,1.000
3,010400,6100,1.000
4,010500,6100,0.446


In [20]:
# Compare against the Dorn's table

dorn_puma2000_cz1990 = pd.read_stata("data/cw_puma2000_czone.dta")
dorn_puma2000_cz1990['puma2000'] = dorn_puma2000_cz1990['puma2000'].astype(str).str.zfill(6)
dorn_puma2000_cz1990['cz1990'] = dorn_puma2000_cz1990['czone'].astype(int)
dorn_puma2000_cz1990 = dorn_puma2000_cz1990[['puma2000', 'cz1990', 'afactor']]

comparison = pd.merge(
    puma2000_cz90_crosswalk,
    dorn_puma2000_cz1990,
    on=['puma2000', 'cz1990'],
    how='outer',
    suffixes=('_mine', '_dorn'),
    indicator=True
)

print(comparison['_merge'].value_counts())
print(f"\nTotal rows: {len(comparison)}")

matched = comparison[comparison['_merge'] == 'both']
print(f"\nMatched pairs: {len(matched)}")
print(f"Mean absolute difference: {(matched['afactor_mine'] - matched['afactor_dorn']).abs().mean():.6f}")
print(f"Max absolute difference: {(matched['afactor_mine'] - matched['afactor_dorn']).abs().max():.6f}")

_merge
both          2880
left_only        0
right_only       0
Name: count, dtype: int64

Total rows: 2880

Matched pairs: 2880
Mean absolute difference: 0.000087
Max absolute difference: 0.002000


In [22]:
puma2000_cz90_crosswalk['puma2000'].str.len().value_counts()

puma2000
6    2880
Name: count, dtype: int64

In [21]:
puma2000_cz90_crosswalk.to_csv('output/puma2000_cz1990_crosswalk.csv', index=False)
puma2000_cz90_crosswalk.to_stata('output/puma2000_cz1990_crosswalk.dta', write_index=False)

## Puma2010 to CZ1990


###  Method: Sequential Crosswalk Method (Dorn's approach)
PUMA 2010 → PUMA 2000 (using Geocorr equivalency) → CZ 1990 (using puma2000_cz90_crosswalk)

In [24]:
# Load PUMA 2010 to PUMA 2000 equivalency file downloaded from Geocorr: https://mcdc.missouri.edu/geography/PUMAs.html

puma2010_to_2000 = pd.read_csv('data/puma2k_puma2010.csv', encoding='latin-1', skiprows=[1])

# Clean and format PUMA codes and allocation factors
puma2010_to_2000['puma12'] = puma2010_to_2000['puma12'].astype(str).str.strip()
puma2010_to_2000['puma2k'] = puma2010_to_2000['puma2k'].astype(str).str.strip()
puma2010_to_2000['AFACT2'] = puma2010_to_2000['AFACT2'].astype(float)

puma2010_to_2000 = puma2010_to_2000[puma2010_to_2000['puma12'] != '']
puma2010_to_2000 = puma2010_to_2000[puma2010_to_2000['puma2k'] != '']

# Create combined PUMA 2010 code (state + puma, 7 digits total)
puma2010_to_2000['puma2010'] = (
    puma2010_to_2000['state'].astype(int).astype(str).str.zfill(2) + 
    puma2010_to_2000['puma12'].astype(int).astype(str).str.zfill(5)
)

# Create combined PUMA 2000 code (state + puma, 6 digits total)
puma2010_to_2000['puma2000'] = (
    puma2010_to_2000['state'].astype(int).astype(str).str.zfill(2) + 
    puma2010_to_2000['puma2k'].astype(int).astype(str).str.zfill(4)
)

# Select final columns and rename allocation factor
puma2010_to_2000 = puma2010_to_2000[['puma2010', 'puma2000', 'AFACT2']].rename(columns={'AFACT2': 'afact'})

# Verify allocation factors sum to 1 for each PUMA 2010
check = puma2010_to_2000.groupby('puma2010')['afact'].sum()
print(check.describe())

count    2351.000000
mean        0.999997
std         0.000237
min         0.999000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.001000
Name: afact, dtype: float64


In [25]:
# Load PUMA 2000 to CZ 1990 crosswalk and format PUMA codes
puma2000_cz90_crosswalk = pd.read_csv('output/puma2000_cz1990_crosswalk.csv')
puma2000_cz90_crosswalk['puma2000'] = puma2000_cz90_crosswalk['puma2000'].astype(str).str.zfill(6)

# Verify PUMA 2000 codes match between the two files
puma2010_puma2000 = set(puma2010_to_2000['puma2000'].unique())
puma2000_crosswalk = set(puma2000_cz90_crosswalk['puma2000'].unique())

only_in_2010to2000 = puma2010_puma2000 - puma2000_crosswalk
only_in_crosswalk = puma2000_crosswalk - puma2010_puma2000

print(f'PUMA 2000 codes only in puma2010_to_2000: {len(only_in_2010to2000)}')
print(f'PUMA 2000 codes only in puma2000_cz90_crosswalk: {len(only_in_crosswalk)}')

if len(only_in_2010to2000) > 0:
    print(f'\nIn puma2010_to_2000 but not in crosswalk:')
    print(sorted(only_in_2010to2000)[:20])

if len(only_in_crosswalk) > 0:
    print(f'\nIn crosswalk but not in puma2010_to_2000:')
    print(sorted(only_in_crosswalk)[:20])

PUMA 2000 codes only in puma2010_to_2000: 0
PUMA 2000 codes only in puma2000_cz90_crosswalk: 0


In [26]:
# Merge PUMA 2010→2000 with PUMA 2000→CZ 1990 to create sequential crosswalk
puma2010_cz90_sequential = pd.merge(
    puma2010_to_2000,
    puma2000_cz90_crosswalk,
    on='puma2000',
    how='left'
)

# Calculate combined allocation factor (product of two-step allocation)
puma2010_cz90_sequential['afact_combined'] = (
    puma2010_cz90_sequential['afact'] * puma2010_cz90_sequential['afactor']
)

# Aggregate to PUMA 2010-CZ level by summing combined allocation factors
puma2010_cz90_sequential_final = (
    puma2010_cz90_sequential.groupby(['puma2010', 'cz1990'])
    .agg({'afact_combined': 'sum'})
    .reset_index()
    .rename(columns={'afact_combined': 'afactor'})
)

puma2010_cz90_sequential_final['afactor'] = puma2010_cz90_sequential_final['afactor'].round(3)

# Verify allocation factors sum to 1 for each PUMA 2010
check = puma2010_cz90_sequential_final.groupby('puma2010')['afactor'].sum()
print(check.describe())

count    2351.000000
mean        0.999997
std         0.000358
min         0.998000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.002000
Name: afactor, dtype: float64


In [27]:
dorn_puma2010_cz1990 = pd.read_stata('data/cw_puma2010_czone.dta')
dorn_puma2010_cz1990['puma2010'] = dorn_puma2010_cz1990['puma2010'].astype(str).str.zfill(7)
dorn_puma2010_cz1990['cz1990'] = dorn_puma2010_cz1990['czone'].astype(int)
dorn_puma2010_cz1990 = dorn_puma2010_cz1990[['puma2010', 'cz1990', 'afactor']]

In [28]:
# Compare sequential crosswalk method with Dorn's published crosswalk
comparison_sequential = pd.merge(
    puma2010_cz90_sequential_final,
    dorn_puma2010_cz1990,
    on=['puma2010', 'cz1990'],
    how='outer',
    suffixes=('_sequential', '_dorn'),
    indicator=True
)

print(comparison_sequential['_merge'].value_counts())
print(f'\nTotal rows: {len(comparison_sequential)}')

matched = comparison_sequential[comparison_sequential['_merge'] == 'both']
print(f'\nMatched pairs: {len(matched)}')
print(f'Mean absolute difference: {(matched["afactor_sequential"] - matched["afactor_dorn"]).abs().mean():.6f}')
print(f'Max absolute difference: {(matched["afactor_sequential"] - matched["afactor_dorn"]).abs().max():.6f}')

_merge
both          4077
left_only        0
right_only       0
Name: count, dtype: int64

Total rows: 4077

Matched pairs: 4077
Mean absolute difference: 0.000178
Max absolute difference: 0.001564


In [29]:
puma2010_cz90_sequential_final.to_csv('output/puma2010_cz1990_crosswalk.csv', index=False)
puma2010_cz90_sequential_final.to_stata('output/puma2010_cz1990_crosswalk.dta', write_index=False)

In [30]:
puma2010_cz90_sequential_final['puma2010'].str.len().value_counts()

puma2010
7    4077
Name: count, dtype: int64

## Puma2020 to CZ1990


###  Method: Sequential Crosswalk Method
PUMA 2020 → PUMA 2010 (using IPUMS equivalency) → CZ 1990 (using puma2010_cz90_crosswalk)

In [32]:
# Load IPUMS PUMA 2010 to PUMA 2020 equivalency file
puma2010_to_2020 = pd.read_excel('data/PUMA2010_PUMA2020_crosswalk.xls')

print('IPUMS file structure:')
print(f'Total rows: {len(puma2010_to_2020)}')

# Create full PUMA codes
puma2010_to_2020['puma2010'] = (
    puma2010_to_2020['State10'].astype(str).str.zfill(2) + 
    puma2010_to_2020['PUMA10'].astype(str).str.zfill(5)
)

puma2010_to_2020['puma2020'] = (
    puma2010_to_2020['State20'].astype(str).str.zfill(2) + 
    puma2010_to_2020['PUMA20'].astype(str).str.zfill(5)
)

# Count unique PUMAs
print(f'\nUnique PUMA 2010 codes: {puma2010_to_2020["puma2010"].nunique()}')
print(f'Unique PUMA 2020 codes: {puma2010_to_2020["puma2020"].nunique()}')

# Count Puerto Rico PUMAs
pr_puma2010 = puma2010_to_2020[puma2010_to_2020['State10'] == 72]['puma2010'].nunique()
pr_puma2020 = puma2010_to_2020[puma2010_to_2020['State20'] == 72]['puma2020'].nunique()

print(f'\nPuerto Rico PUMA 2010 codes: {pr_puma2010}')
print(f'Puerto Rico PUMA 2020 codes: {pr_puma2020}')

# Mainland only
mainland_puma2010 = puma2010_to_2020[puma2010_to_2020['State10'] != 72]['puma2010'].nunique()
mainland_puma2020 = puma2010_to_2020[puma2010_to_2020['State20'] != 72]['puma2020'].nunique()

print(f'\nMainland PUMA 2010 codes (excluding PR): {mainland_puma2010}')
print(f'Mainland PUMA 2020 codes (excluding PR): {mainland_puma2020}')


IPUMS file structure:
Total rows: 6970

Unique PUMA 2010 codes: 2378
Unique PUMA 2020 codes: 2486

Puerto Rico PUMA 2010 codes: 27
Puerto Rico PUMA 2020 codes: 24

Mainland PUMA 2010 codes (excluding PR): 2351
Mainland PUMA 2020 codes (excluding PR): 2462


In [33]:
# Verify it sums to 1 for each PUMA 2020
check = puma2010_to_2020.groupby('puma2020')['pPUMA20_Pop20'].sum()
print('Verification that pPUMA20_Pop20 sums to ~100 for each PUMA 2020:')
print((check / 100).describe())

Verification that pPUMA20_Pop20 sums to ~100 for each PUMA 2020:
count    2486.000000
mean        0.999999
std         0.000023
min         0.999900
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000100
Name: pPUMA20_Pop20, dtype: float64


In [34]:
# Exclude Puerto Rico (not in CZ 1990 mapping)
puma2010_to_2020 = puma2010_to_2020[puma2010_to_2020['State10'] != 72]

# Prepare IPUMS data with allocation factor
puma2010_to_2020_clean = puma2010_to_2020[['puma2020', 'puma2010', 'pPUMA20_Pop20']].copy()
puma2010_to_2020_clean['afact'] = puma2010_to_2020_clean['pPUMA20_Pop20'] / 100  # Convert percent to fraction
puma2010_to_2020_clean = puma2010_to_2020_clean[['puma2020', 'puma2010', 'afact']]

# Verify allocation factors sum to 1 for each PUMA 2020
check = puma2010_to_2020_clean.groupby('puma2020')['afact'].sum()
print('Verification that allocation factors sum to 1 for each PUMA 2020:')
print(check.describe())

Verification that allocation factors sum to 1 for each PUMA 2020:
count    2462.000000
mean        0.999999
std         0.000023
min         0.999900
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000100
Name: afact, dtype: float64


In [35]:
# Load PUMA 2010 to CZ 1990 crosswalk (sequential method)
puma2010_cz90_sequential_final = pd.read_csv('output/puma2010_cz1990_crosswalk.csv')
puma2010_cz90_sequential_final['puma2010'] = puma2010_cz90_sequential_final['puma2010'].astype(str).str.zfill(7)

# Verify PUMA 2010 codes match between the two files
puma2020_puma2010 = set(puma2010_to_2020_clean['puma2010'].unique())
puma2010_crosswalk = set(puma2010_cz90_sequential_final['puma2010'].unique())

only_in_ipums = puma2020_puma2010 - puma2010_crosswalk
only_in_crosswalk = puma2010_crosswalk - puma2020_puma2010

print(f'PUMA 2010 codes only in IPUMS: {len(only_in_ipums)}')
print(f'PUMA 2010 codes only in crosswalk: {len(only_in_crosswalk)}')

if len(only_in_ipums) > 0:
    print(f'\nIn IPUMS but not in crosswalk:')
    print(sorted(only_in_ipums)[:20])

if len(only_in_crosswalk) > 0:
    print(f'\nIn crosswalk but not in IPUMS:')
    print(sorted(only_in_crosswalk)[:20])

PUMA 2010 codes only in IPUMS: 0
PUMA 2010 codes only in crosswalk: 0


In [36]:
# Merge PUMA 2020→2010 with PUMA 2010→CZ 1990 to create sequential crosswalk
puma2020_cz90_sequential = pd.merge(
    puma2010_to_2020_clean,
    puma2010_cz90_sequential_final,
    on='puma2010',
    how='left'
)

print(f'Null CZ codes: {puma2020_cz90_sequential["cz1990"].isnull().sum()}')

# Calculate combined allocation factor (product of two-step allocation)
puma2020_cz90_sequential['afact_combined'] = (
    puma2020_cz90_sequential['afact'] * puma2020_cz90_sequential['afactor']
)

# Aggregate to PUMA 2020-CZ level by summing combined allocation factors
puma2020_cz90_sequential_final = (
    puma2020_cz90_sequential.groupby(['puma2020', 'cz1990'])
    .agg({'afact_combined': 'sum'})
    .reset_index()
    .rename(columns={'afact_combined': 'afactor'})
)

puma2020_cz90_sequential_final['afactor'] = puma2020_cz90_sequential_final['afactor'].round(3)

# Verify allocation factors sum to 1 for each PUMA 2020
check = puma2020_cz90_sequential_final.groupby('puma2020')['afactor'].sum()
print('\nVerification:')
print(check.describe())

Null CZ codes: 0

Verification:
count    2462.000000
mean        0.999998
std         0.000398
min         0.998000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.004000
Name: afactor, dtype: float64


In [37]:
puma2020_cz90_sequential_final.to_csv('output/puma2020_cz1990_crosswalk.csv', index=False)
puma2020_cz90_sequential_final.to_stata('output/puma2020_cz1990_crosswalk.dta', write_index=False)

In [38]:
puma2020_cz90_sequential_final['puma2020'].str.len().value_counts()

puma2020
7    6757
Name: count, dtype: int64